In [ ]:
# pip install earthengine-api

   ---------------------------------------- 0.0/16.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.8 MB ? eta -:--:--
    --------------------------------------- 0.3/16.8 MB ? eta -:--:--
    --------------------------------------- 0.3/16.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.8 MB 730.2 kB/s eta 0:00:23
   - -------------------------------------- 0.5/16.8 MB 730.2 kB/s eta 0:00:23
   - -------------------------------------- 0.8/16.8 MB 699.0 kB/s eta 0:00:23
   - -------------------------------------- 0.8/16.8 MB 699.0 kB/s eta 0:00:23
   -- ------------------------------------- 1.0/16.8 MB 680.3 kB/s eta 0:00:24
   --- ------------------------------------ 1.3/16.8 MB 684.9 kB/s eta 0:00:23
   --- ------------------------------------ 1.3/16.8 MB 684.9 kB/s eta 0:00:23
   --- ------------------------------------ 1.6/16.8 MB 671.1 kB/s eta 0:00:23
   --- ------------------------------------ 1.6/16.8 MB 671.1 kB/s eta 0:00:23
   ---- -

In [1]:
import ee
import pandas as pd

In [3]:
ee.Authenticate()


Successfully saved authorization token.


In [4]:
ee.Initialize(project='multimodal-flood-risk')

print("Earth Engine initialized successfully.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized successfully.


## Define the 2 areas

In [5]:
# ------------------------------------------------------------
# PP1 Candidate AOI
# ------------------------------------------------------------

candidate_aoi = ee.Geometry.Rectangle([
    79.82,   # west
    6.80,    # south
    80.28,   # east
    7.25     # north
])


# ------------------------------------------------------------
# Small technical pilot AOI
# ------------------------------------------------------------

pilot_aoi = ee.Geometry.Rectangle([
    80.055,  # west
    6.885,  # south
    80.105, # east
    6.935   # north
])


print(
    "Candidate AOI area (km²):",
    candidate_aoi.area(1).divide(1_000_000).getInfo()
)

print(
    "Hanwella pilot area (km²):",
    pilot_aoi.area(1).divide(1_000_000).getInfo()
)

Candidate AOI area (km²): 2540.210940965659
Hanwella pilot area (km²): 30.686324537958345


## Defining the EV2 search window

In [6]:
SEARCH_START = '2025-11-20'
SEARCH_END = '2025-12-24'

print("Sentinel-1 search window:")
print(SEARCH_START, "to", SEARCH_END)

Sentinel-1 search window:
2025-11-20 to 2025-12-24


## Retrieving Sentinel-1

In [7]:
s1_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(candidate_aoi)
    .filterDate(SEARCH_START, SEARCH_END)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(
        ee.Filter.listContains(
            'transmitterReceiverPolarisation',
            'VV'
        )
    )
    .filter(
        ee.Filter.listContains(
            'transmitterReceiverPolarisation',
            'VH'
        )
    )
    .sort('system:time_start')
)

s1_count = s1_collection.size().getInfo()

print("Number of Sentinel-1 candidate scenes:", s1_count)

Number of Sentinel-1 candidate scenes: 14


## Sentinel‑1 Metadata Inventory

In [8]:
def extract_s1_metadata(image):
    """
    Extract the metadata needed for PP1 Sentinel-1
    acquisition screening.
    """

    acquisition_time = ee.Date(
        image.get('system:time_start')
    )

    intersection_area = (
        image.geometry()
        .intersection(candidate_aoi, 1)
        .area(1)
    )

    candidate_area = candidate_aoi.area(1)

    coverage_percent = (
        intersection_area
        .divide(candidate_area)
        .multiply(100)
    )

    return ee.Feature(None, {

        'scene_id':
            image.get('system:index'),

        'asset_id':
            image.id(),

        'acquisition_utc':
            acquisition_time.format(
                'yyyy-MM-dd HH:mm',
                'UTC'
            ),

        'acquisition_local':
            acquisition_time.format(
                'yyyy-MM-dd HH:mm',
                'Asia/Colombo'
            ),

        'platform':
            image.get('platform_number'),

        'orbit_pass':
            image.get(
                'orbitProperties_pass'
            ),

        'relative_orbit':
            image.get(
                'relativeOrbitNumber_start'
            ),

        'instrument_mode':
            image.get(
                'instrumentMode'
            ),

        'polarization':
            ee.List(
                image.get(
                    'transmitterReceiverPolarisation'
                )
            ).join(','),

        'resolution_m':
            image.get(
                'resolution_meters'
            ),

        'slice_number':
            image.get(
                'sliceNumber'
            ),

        'total_slices':
            image.get(
                'totalSlices'
            ),

        'coverage_percent':
            coverage_percent

    })

In [9]:
inventory_fc = ee.FeatureCollection(
    s1_collection.map(extract_s1_metadata)
)

inventory_data = inventory_fc.getInfo()

rows = [
    feature['properties']
    for feature in inventory_data['features']
]

inventory_df = pd.DataFrame(rows)

inventory_df

,acquisition_local,acquisition_utc,asset_id,coverage_percent,instrument_mode,orbit_pass,platform,polarization,relative_orbit,resolution_m,scene_id,slice_number,total_slices
0,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,56.693060,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,2,3
1,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,43.373912,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,3,3
2,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,6.576538,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,1,3
3,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,3.084946,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,2,3
4,2025-12-02 05:54,2025-12-02 00:24,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,100.000000,IW,DESCENDING,C,"VV,VH",19,10,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,3,4
5,2025-12-02 18:18,2025-12-02 12:48,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,9.728797,IW,ASCENDING,C,"VV,VH",27,10,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,2,6
6,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,56.733736,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,2,3
7,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,43.333318,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,3,3
8,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,6.686733,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,1,3
9,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,3.108311,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,2,3


In [10]:
inventory_df = inventory_df.sort_values(
    'acquisition_local'
).reset_index(drop=True)

inventory_df

,acquisition_local,acquisition_utc,asset_id,coverage_percent,instrument_mode,orbit_pass,platform,polarization,relative_orbit,resolution_m,scene_id,slice_number,total_slices
0,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,56.693060,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,2,3
1,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,43.373912,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,3,3
2,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,6.576538,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,1,3
3,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,3.084946,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,2,3
4,2025-12-02 05:54,2025-12-02 00:24,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,100.000000,IW,DESCENDING,C,"VV,VH",19,10,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,3,4
5,2025-12-02 18:18,2025-12-02 12:48,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,9.728797,IW,ASCENDING,C,"VV,VH",27,10,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,2,6
6,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,56.733736,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,2,3
7,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,43.333318,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,3,3
8,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,6.686733,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,1,3
9,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,3.108311,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,2,3


In [11]:
inventory_df[
    [
        'scene_id',
        'acquisition_local',
        'platform',
        'orbit_pass',
        'relative_orbit',
        'instrument_mode',
        'polarization',
        'coverage_percent'
    ]
]

,scene_id,acquisition_local,platform,orbit_pass,relative_orbit,instrument_mode,polarization,coverage_percent
0,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,2025-11-26 05:55,A,DESCENDING,19,IW,"VV,VH",56.693060
1,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,2025-11-26 05:55,A,DESCENDING,19,IW,"VV,VH",43.373912
2,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,2025-11-26 18:19,A,ASCENDING,27,IW,"VV,VH",6.576538
3,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,2025-11-26 18:19,A,ASCENDING,27,IW,"VV,VH",3.084946
4,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,2025-12-02 05:54,C,DESCENDING,19,IW,"VV,VH",100.000000
5,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,2025-12-02 18:18,C,ASCENDING,27,IW,"VV,VH",9.728797
6,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,2025-12-08 05:55,A,DESCENDING,19,IW,"VV,VH",56.733736
7,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,2025-12-08 05:55,A,DESCENDING,19,IW,"VV,VH",43.333318
8,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,2025-12-08 18:19,A,ASCENDING,27,IW,"VV,VH",6.686733
9,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,2025-12-08 18:19,A,ASCENDING,27,IW,"VV,VH",3.108311


In [12]:
inventory_df.insert(
    0,
    'event_id',
    'EV2_Nov_2025'
)

inventory_df.insert(
    1,
    'aoi_id',
    'PP1_AOI_KELANI_01'
)

inventory_df

,event_id,aoi_id,acquisition_local,acquisition_utc,asset_id,coverage_percent,instrument_mode,orbit_pass,platform,polarization,relative_orbit,resolution_m,scene_id,slice_number,total_slices
0,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,56.693060,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002515_20251126T0025...,2,3
1,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-11-26 05:55,2025-11-26 00:25,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,43.373912,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251126T002540_20251126T0026...,3,3
2,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,6.576538,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124916_20251126T1249...,1,3
3,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-11-26 18:19,2025-11-26 12:49,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,3.084946,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251126T124945_20251126T1250...,2,3
4,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-02 05:54,2025-12-02 00:24,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,100.000000,IW,DESCENDING,C,"VV,VH",19,10,S1C_IW_GRDH_1SDV_20251202T002429_20251202T0024...,3,4
5,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-02 18:18,2025-12-02 12:48,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,9.728797,IW,ASCENDING,C,"VV,VH",27,10,S1C_IW_GRDH_1SDV_20251202T124832_20251202T1248...,2,6
6,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,56.733736,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002514_20251208T0025...,2,3
7,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-08 05:55,2025-12-08 00:25,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,43.333318,IW,DESCENDING,A,"VV,VH",19,10,S1A_IW_GRDH_1SDV_20251208T002539_20251208T0026...,3,3
8,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,6.686733,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124915_20251208T1249...,1,3
9,EV2_Nov_2025,PP1_AOI_KELANI_01,2025-12-08 18:19,2025-12-08 12:49,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,3.108311,IW,ASCENDING,A,"VV,VH",27,10,S1A_IW_GRDH_1SDV_20251208T124944_20251208T1250...,2,3


## Saving the inventory

In [15]:
inventory_path = (
    "../shared/events/s1_scene_inventory_ev2_pp1_bt_mask_pilot.csv"
)

inventory_df.to_csv(
    inventory_path,
    index=False
)

print(
    f"Saved Sentinel-1 inventory to: {inventory_path}"
)

Saved Sentinel-1 inventory to: ../shared/events/s1_scene_inventory_ev2_pp1_bt_mask_pilot.csv
